In [ ]:
import pandas as pd
import numpy as np
import json


FileNotFoundError: File trains.json does not exist

In [4]:
with open('stations.json') as f:
    stations = json.load(f)

with open('trains.json') as f:
    trains = json.load(f)

with open('schedules.json') as f:
    schedules = json.load(f)

In [5]:
stations_df = pd.json_normalize(stations['features'])
trains_df = pd.DataFrame(trains)
schedules_df = pd.DataFrame(schedules)

stations_df.head()

,type,geometry.type,geometry.coordinates,properties.state,properties.code,properties.name,properties.zone,properties.address,geometry
0,Feature,Point,"[75.4516454, 27.2520587]",Rajasthan,BDHL,Badhal,NWR,"Kishangarh Renwal, Rajasthan",NaN
1,Feature,NaN,NaN,None,XX-BECE,XX-BECE,None,None,NaN
2,Feature,NaN,NaN,None,XX-BSPY,XX-BSPY,None,None,NaN
3,Feature,NaN,NaN,None,YY-BPLC,YY-BPLC,None,None,NaN
4,Feature,Point,"[79.519746, 28.913427000000002]",Uttar Pradesh,KHH,KICHHA,NER,"Kichha, Uttar Pradesh",NaN


In [6]:
print(stations_df.shape)
print(trains_df.shape)
print(schedules_df.shape)

stations_df.info()

(8990, 9)
(5208, 2)
(417080, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8990 entries, 0 to 8989
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  8990 non-null   object 
 1   geometry.type         8697 non-null   object 
 2   geometry.coordinates  8697 non-null   object 
 3   properties.state      4458 non-null   object 
 4   properties.code       8990 non-null   object 
 5   properties.name       8990 non-null   object 
 6   properties.zone       4458 non-null   object 
 7   properties.address    4458 non-null   object 
 8   geometry              0 non-null      float64
dtypes: float64(1), object(8)
memory usage: 632.2+ KB


In [7]:
stations_df = stations_df[['properties.code', 'properties.name']]
stations_df.columns = ['code', 'name']
stations_df.dropna(inplace=True)

stations_df.head()

,code,name
0,BDHL,Badhal
1,XX-BECE,XX-BECE
2,XX-BSPY,XX-BSPY
3,YY-BPLC,YY-BPLC
4,KHH,KICHHA


In [8]:
schedules_df.dropna(inplace=True)

schedules_df = schedules_df[schedules_df['station_code'] != '0']

In [9]:
schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], errors='coerce')
schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], errors='coerce')

/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_19424/1558200334.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], errors='coerce')
/var/folders/jq/7mz7h2f9643255w992tsk2140000gn/T/ipykernel_19424/1558200334.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], errors='coerce')


In [10]:
schedules_df['arrival'].head()

0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: arrival, dtype: datetime64[ns]

In [11]:
schedules_df['arrival'] = pd.to_datetime(schedules_df['arrival'], format='%H:%M:%S', errors='coerce')
schedules_df['departure'] = pd.to_datetime(schedules_df['departure'], format='%H:%M:%S', errors='coerce')

In [12]:
schedules_df['delay'] = (schedules_df['departure'] - schedules_df['arrival']).dt.total_seconds()

In [13]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
schedules_df['station_code'] = le.fit_transform(schedules_df['station_code'])

In [14]:
df = schedules_df.merge(trains_df, on='train_number', how='left')
df = df.merge(stations_df, left_on='station_code', right_on='code', how='left')

KeyError: 'train_number'

In [ ]:
df.drop_duplicates(inplace=True)
df.fillna(0, inplace=True)

In [15]:
df.to_csv("cleaned_railway_data.csv", index=False)

NameError: name 'df' is not defined

In [16]:
print(schedules_df.columns)
print(trains_df.columns)

Index(['arrival', 'day', 'train_name', 'station_name', 'station_code', 'id',
       'train_number', 'departure', 'delay'],
      dtype='object')
Index(['type', 'features'], dtype='object')


In [17]:
trains_df = pd.json_normalize(trains['features'])
trains_df.head()

,type,geometry.type,geometry.coordinates,properties.third_ac,properties.arrival,properties.from_station_code,properties.name,properties.zone,properties.chair_car,properties.first_class,...,properties.departure,properties.return_train,properties.to_station_code,properties.second_ac,properties.classes,properties.to_station_name,properties.duration_h,properties.type,properties.first_ac,properties.distance
0,Feature,LineString,"[[74.880117, 32.706975], [74.953339, 32.762368...",0,12:15:00,JAT,Jammu Tawi Udhampur Special,NR,0,0,...,10:40:00,04602,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
1,Feature,LineString,"[[75.154881, 32.92664], [75.14542599999999, 32...",0,08:35:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,...,06:45:00,04601,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
2,Feature,LineString,"[[74.880117, 32.706975], [74.953339, 32.762368...",0,17:50:00,JAT,JAT UDAHMPUR DMU,NR,0,0,...,16:15:00,04604,UHP,0,,UDHAMPUR,1.0,DEMU,0,53.0
3,Feature,LineString,"[[75.154881, 32.92664], [75.14542599999999, 32...",0,19:50:00,UHP,UDHAMPUR JAMMUTAWI DMU,NR,0,0,...,18:20:00,04603,JAT,0,,JAMMU TAWI,1.0,DEMU,0,53.0
4,Feature,LineString,"[[72.840535, 19.061911], [72.840078, 19.069166...",1,12:30:00,BDTS,Mumbai BandraT-Bikaner SF Special,NWR,0,0,...,14:35:00,04727,BKN,1,,BIKANER JN,21.0,SF,0,1212.0


In [18]:
print(trains_df.columns)


Index(['type', 'geometry.type', 'geometry.coordinates', 'properties.third_ac',
       'properties.arrival', 'properties.from_station_code', 'properties.name',
       'properties.zone', 'properties.chair_car', 'properties.first_class',
       'properties.duration_m', 'properties.sleeper',
       'properties.from_station_name', 'properties.number',
       'properties.departure', 'properties.return_train',
       'properties.to_station_code', 'properties.second_ac',
       'properties.classes', 'properties.to_station_name',
       'properties.duration_h', 'properties.type', 'properties.first_ac',
       'properties.distance'],
      dtype='object')


In [19]:
trains_df = trains_df[['properties.number', 'properties.name']]
trains_df.columns = ['train_number', 'train_name']

In [20]:
df = schedules_df.merge(trains_df, on='train_number', how='left')

In [21]:
df = df.merge(stations_df, left_on='station_code', right_on='code', how='left')

ValueError: You are trying to merge on int64 and object columns for key 'station_code'. If you wish to proceed you should use pd.concat

In [ ]:
df['hour'] = pd.to_datetime(df['departure']).dt.hour
df['day_of_week'] = pd.to_datetime(df['departure']).dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

In [ ]:
df['station_traffic'] = df.groupby('station_code')['train_number'].transform('count')

In [11]:
df.columns

NameError: name 'df' is not defined